<a href="https://colab.research.google.com/github/wuqichuang/learning-python/blob/main/notebooks/06_day6-15_project_solution.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 在浏览器中打开本 notebook 后，点击菜单 **代码执行程序 → 全部运行** 即可按顺序执行所有单元格。

# 参考实现｜学生管理系统（Day 6–15 对照用）

> ⚠️ **使用方法**：先按任务书自己动手做，卡住 30 分钟以上再来看这里。看懂后**合上它自己重写一遍**——抄一遍和写一遍的效果天差地别。

**关于数据库的说明**：本 notebook 里的代码默认用 **sqlite**（Colab 可直接运行调试，无需服务器）。
正式部署到服务器时只需改两处（Day 14 会操作）：
1. 连接：`sqlite3.connect(...)` → `pymysql.connect(host=..., user=..., password=..., database=..., charset='utf8mb4')`
2. 参数占位符：sqlite 用 `?`，MySQL(pymysql) 用 `%s`

其余 SQL 和 Python 逻辑**完全一样**。每个函数都按"真实项目"的标准写，带注释。

---
## 0. 建库建表 + 测试数据（对应 Day 6）

In [ ]:
import sqlite3
import pandas as pd

DB = "student_mgmt.db"   # sqlite 数据库文件；MySQL 版替换为 pymysql 连接

def get_conn():
    """获取数据库连接。MySQL 版：
    import pymysql
    return pymysql.connect(host='你的服务器IP', user='learn',
                           password='Learn@2026', database='student_mgmt', charset='utf8mb4')
    """
    return sqlite3.connect(DB)

def init_db():
    """建三张表：班级、学生、成绩（MySQL 版把 INTEGER PRIMARY KEY AUTOINCREMENT
    换成 INT PRIMARY KEY AUTO_INCREMENT，其余相同）"""
    conn = get_conn()
    conn.executescript("""
    DROP TABLE IF EXISTS scores;
    DROP TABLE IF EXISTS students;
    DROP TABLE IF EXISTS classes;
    CREATE TABLE classes (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        cls_name TEXT NOT NULL UNIQUE,
        teacher TEXT
    );
    CREATE TABLE students (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        cls_id INTEGER REFERENCES classes(id),
        gender TEXT,
        age INTEGER
    );
    CREATE TABLE scores (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        stu_id INTEGER NOT NULL REFERENCES students(id),
        subject TEXT NOT NULL,
        score REAL,
        UNIQUE(stu_id, subject)
    );
    """)
    conn.commit()
    conn.close()
    print("✅ 建表完成：classes / students / scores")

init_db()

In [ ]:
def seed_data():
    """插入测试数据：2 个班、6 名学生、每人 2–3 科成绩"""
    conn = get_conn()
    cur = conn.cursor()
    cur.executemany("INSERT INTO classes (cls_name, teacher) VALUES (?, ?)",
                    [("一班", "张老师"), ("二班", "王老师")])
    students = [("李雷", 1, "男", 17), ("韩梅梅", 2, "女", 17), ("Jim", 1, "男", 18),
                ("Lucy", 2, "女", 17), ("Lily", 1, "女", 18), ("王强", 2, "男", 17)]
    cur.executemany("INSERT INTO students (name, cls_id, gender, age) VALUES (?, ?, ?, ?)", students)
    scores = [(1,"数学",88),(1,"英语",92),(1,"语文",85),
              (2,"数学",91),(2,"英语",84),
              (3,"数学",76),(3,"英语",88),(3,"语文",90),
              (4,"数学",59),(4,"英语",71),
              (5,"数学",95),(5,"英语",60),(5,"语文",78),
              (6,"数学",83),(6,"英语",79)]
    cur.executemany("INSERT INTO scores (stu_id, subject, score) VALUES (?, ?, ?)", scores)
    conn.commit()
    print(f"✅ 测试数据：{len(students)} 名学生，{len(scores)} 条成绩")
    conn.close()

seed_data()

---
## 1. 查询功能（对应 Day 8）

三个查询 + 一个共用的"打印表格"工具函数。

In [ ]:
def print_table(rows, headers):
    """用 f-string 对齐打印查询结果（Day 1 技能）"""
    if not rows:
        print("（未找到记录）")
        return
    widths = [max(len(str(h)), *(len(str(r[i])) for r in rows)) for i, h in enumerate(headers)]
    line = "  ".join(str(h).ljust(w) for h, w in zip(headers, widths))
    print(line); print("-" * len(line))
    for r in rows:
        print("  ".join(str(x).ljust(w) for x, w in zip(r, widths)))
    print(f"（共 {len(rows)} 条记录）")

def show_all():
    """查看全部学生（JOIN 班级表）"""
    conn = get_conn()
    rows = conn.execute("""
        SELECT s.id, s.name, c.cls_name, s.gender, s.age
        FROM students s JOIN classes c ON s.cls_id = c.id ORDER BY s.id""").fetchall()
    conn.close()
    print_table(rows, ["学号", "姓名", "班级", "性别", "年龄"])

def search_by_name(keyword):
    """按姓名模糊查询。⚠️ 用户输入一律走占位符参数，绝不拼进 SQL 字符串！"""
    conn = get_conn()
    rows = conn.execute("""
        SELECT s.id, s.name, c.cls_name FROM students s
        JOIN classes c ON s.cls_id = c.id WHERE s.name LIKE ?""",
        (f"%{keyword}%",)).fetchall()
    conn.close()
    print_table(rows, ["学号", "姓名", "班级"])

def search_by_cls(cls_name):
    """按班级查询该班所有学生"""
    conn = get_conn()
    rows = conn.execute("""
        SELECT s.id, s.name, s.gender, s.age FROM students s
        JOIN classes c ON s.cls_id = c.id WHERE c.cls_name = ?""",
        (cls_name,)).fetchall()
    conn.close()
    print_table(rows, ["学号", "姓名", "性别", "年龄"])

show_all()
print()
search_by_name("李")
print()
search_by_cls("一班")

## 2. 添加学生（对应 Day 7）

In [ ]:
def add_student(name, cls_name, gender, age):
    """添加学生：班级名不存在则自动建班；返回新学号"""
    if not name.strip():
        print("⚠️ 姓名不能为空"); return None
    conn = get_conn(); cur = conn.cursor()
    # 班级名 → cls_id（没有就创建）
    cur.execute("INSERT OR IGNORE INTO classes (cls_name) VALUES (?)", (cls_name,))
    # MySQL 版：INSERT IGNORE INTO classes ...
    cls_id = cur.execute("SELECT id FROM classes WHERE cls_name = ?", (cls_name,)).fetchone()[0]
    cur.execute("INSERT INTO students (name, cls_id, gender, age) VALUES (?, ?, ?, ?)",
                (name, cls_id, gender, age))
    new_id = cur.lastrowid
    conn.commit(); conn.close()
    print(f"✅ 添加成功，学号为 {new_id}")
    return new_id

add_student("赵欣", "三班", "女", 17)   # 三班不存在，会自动创建
show_all()

## 3. 修改成绩与删除学生（对应 Day 9）

In [ ]:
def update_score(stu_id, subject, new_score, confirm=True):
    """修改成绩：先查旧值，确认后再改（confirm 模拟用户输入 yes）"""
    if not (0 <= new_score <= 100):
        print("⚠️ 分数必须在 0–100 之间"); return
    conn = get_conn()
    row = conn.execute("SELECT score FROM scores WHERE stu_id=? AND subject=?",
                       (stu_id, subject)).fetchone()
    if row is None:
        print("⚠️ 未找到该学生的该科成绩"); conn.close(); return
    print(f"当前成绩：{row[0]} → 将修改为：{new_score}")
    if confirm:   # 交互版：confirm = input("确认修改？(yes/n) ").strip() == "yes"
        conn.execute("UPDATE scores SET score=? WHERE stu_id=? AND subject=?",
                     (new_score, stu_id, subject))
        conn.commit(); print("✅ 修改完成")
    else:
        print("已取消")
    conn.close()

def delete_student(stu_id, confirm=True):
    """删除学生：先删成绩再删学生（外键顺序），确认后执行"""
    conn = get_conn()
    stu = conn.execute("SELECT name FROM students WHERE id=?", (stu_id,)).fetchone()
    if stu is None:
        print("⚠️ 学号不存在"); conn.close(); return
    print(f"将删除学生：{stu[0]}（学号 {stu_id}）及其全部成绩")
    if confirm:   # 交互版同样用 input 确认
        conn.execute("DELETE FROM scores WHERE stu_id=?", (stu_id,))
        conn.execute("DELETE FROM students WHERE id=?", (stu_id,))
        conn.commit(); print("✅ 删除完成")
    else:
        print("已取消，数据未变")
    conn.close()

update_score(4, "数学", 65)      # Lucy 数学 59 → 65
delete_student(7, confirm=False) # 演示"取消"分支
delete_student(7)                # 删除赵欣
show_all()

## 4. 统计报表（对应 Day 10–11）

In [ ]:
def report_subject_avg():
    """各科平均分（含及格率、优秀率）"""
    conn = get_conn()
    df = pd.read_sql("""
        SELECT subject AS 科目,
               ROUND(AVG(score),1) AS 平均分,
               MAX(score) AS 最高分,
               ROUND(100.0*SUM(score>=60)/COUNT(*),1) AS '及格率%',
               ROUND(100.0*SUM(score>=90)/COUNT(*),1) AS '优秀率%'
        FROM scores GROUP BY subject""", conn)
    conn.close()
    print(df.to_string(index=False))

def report_top(n=5):
    """总分排名榜"""
    conn = get_conn()
    df = pd.read_sql("""
        SELECT s.name AS 姓名, c.cls_name AS 班级,
               SUM(sc.score) AS 总分, ROUND(AVG(sc.score),1) AS 平均分
        FROM scores sc
        JOIN students s ON sc.stu_id = s.id
        JOIN classes  c ON s.cls_id  = c.id
        GROUP BY s.id ORDER BY 总分 DESC LIMIT ?""", conn, params=(n,))
    conn.close()
    df.insert(0, "名次", range(1, len(df)+1))
    print(df.to_string(index=False))

report_subject_avg()
print()
report_top(5)

In [ ]:
def report_cls_compare():
    """班级×科目 平均分交叉表（pivot_table）"""
    conn = get_conn()
    df = pd.read_sql("""
        SELECT c.cls_name AS 班级, sc.subject AS 科目, sc.score
        FROM scores sc
        JOIN students s ON sc.stu_id = s.id
        JOIN classes  c ON s.cls_id  = c.id""", conn)
    conn.close()
    pivot = df.pivot_table(index="班级", columns="科目", values="score", aggfunc="mean").round(1)
    print(pivot.to_string())

report_cls_compare()

## 5. CSV 批量导入 / 导出（对应 Day 12–13）

In [ ]:
def import_csv(path):
    """批量导入：姓名,班级,性别,年龄,科目,分数
    - 班级/学生不存在自动创建
    - 分数非法跳过并警告
    - 同学生同科目已存在则更新（防重复）"""
    df = pd.read_csv(path)
    conn = get_conn(); cur = conn.cursor()
    ok, skip = 0, 0
    for _, r in df.iterrows():
        if not (0 <= r["分数"] <= 100):
            print(f"⚠️ 跳过非法分数：{r['姓名']} {r['科目']} {r['分数']}")
            skip += 1; continue
        cur.execute("INSERT OR IGNORE INTO classes (cls_name) VALUES (?)", (r["班级"],))
        cls_id = cur.execute("SELECT id FROM classes WHERE cls_name=?", (r["班级"],)).fetchone()[0]
        cur.execute("""INSERT INTO students (name, cls_id, gender, age)
                       SELECT ?,?,?,? WHERE NOT EXISTS
                       (SELECT 1 FROM students WHERE name=? AND cls_id=?)""",
                    (r["姓名"], cls_id, r["性别"], r["年龄"], r["姓名"], cls_id))
        stu_id = cur.execute("SELECT id FROM students WHERE name=? AND cls_id=?",
                             (r["姓名"], cls_id)).fetchone()[0]
        # MySQL 版：INSERT ... ON DUPLICATE KEY UPDATE score=VALUES(score)
        cur.execute("""INSERT INTO scores (stu_id, subject, score) VALUES (?,?,?)
                       ON CONFLICT(stu_id, subject) DO UPDATE SET score=excluded.score""",
                    (stu_id, r["科目"], r["分数"]))
        ok += 1
    conn.commit(); conn.close()
    print(f"✅ 导入完成：成功 {ok} 条，跳过 {skip} 条")

# 造一个演示 CSV（含 1 条非法数据），连续导入两次验证防重复
demo_csv = """姓名,班级,性别,年龄,科目,分数
孙倩,一班,女,17,数学,91
孙倩,一班,女,17,英语,88
周杰,二班,男,18,数学,72
周杰,二班,男,18,英语,68
吴迪,三班,男,17,数学,150
吴迪,三班,男,17,英语,83
李雷,一班,男,17,数学,90
"""
with open("import_demo.csv", "w", encoding="utf-8-sig") as f:
    f.write(demo_csv)

import_csv("import_demo.csv")
print("--- 重复导入同一文件（应全部更新而非报错）---")
import_csv("import_demo.csv")

In [ ]:
def student_report(stu_id):
    """生成单个学生的文字成绩报告（f-string 综合应用）"""
    conn = get_conn()
    stu = conn.execute("""
        SELECT s.name, c.cls_name FROM students s
        JOIN classes c ON s.cls_id=c.id WHERE s.id=?""", (stu_id,)).fetchone()
    if stu is None:
        print("⚠️ 学号不存在"); conn.close(); return None
    name, cls_name = stu
    rows = conn.execute("""
        SELECT sc.subject, sc.score,
               (SELECT ROUND(AVG(score),1) FROM scores sc2
                JOIN students s2 ON sc2.stu_id=s2.id
                WHERE sc2.subject=sc.subject AND s2.cls_id=
                      (SELECT cls_id FROM students WHERE id=?)) AS cls_avg
        FROM scores sc WHERE sc.stu_id=?""", (stu_id, stu_id)).fetchall()
    total = sum(r[1] for r in rows)
    rank = conn.execute("""
        SELECT COUNT(*)+1 FROM (
            SELECT s2.id, SUM(sc2.score) t FROM scores sc2
            JOIN students s2 ON sc2.stu_id=s2.id
            WHERE s2.cls_id=(SELECT cls_id FROM students WHERE id=?)
            GROUP BY s2.id) WHERE t > ?""", (stu_id, total)).fetchone()[0]
    n_cls = conn.execute("SELECT COUNT(*) FROM students WHERE cls_id=(SELECT cls_id FROM students WHERE id=?)",
                         (stu_id,)).fetchone()[0]
    conn.close()
    comment = "综合成绩优秀，保持！" if total/len(rows) >= 85 else \
              "成绩良好，继续加油。" if total/len(rows) >= 70 else \
              "成绩及格，仍有提升空间。" if total/len(rows) >= 60 else "需要重点关注，加油！"
    lines = ["="*30, f"姓名：{name}    班级：{cls_name}    学号：{stu_id}", "-"*30]
    for subj, score, avg in rows:
        cmp = "超过平均" if score >= avg else "低于平均"
        lines.append(f"{subj}：{score:.0f} 分（班级平均 {avg}，{cmp}）")
    lines += ["-"*30, f"总分：{total:.0f}，班级排名：{rank} / {n_cls}", f"评语：{comment}", "="*30]
    report = "\n".join(lines)
    return report

r = student_report(1)
print(r)
with open("report_1.txt", "w", encoding="utf-8") as f:
    f.write(r)
print("\n✅ 已保存 report_1.txt")

## 6. 交互菜单与完整 app.py（对应 Day 7/14）

下面单元格把**全部功能 + 交互菜单**整理成单个 `app.py` 文件，可直接 `scp` 上传到服务器运行。
服务器版把 `get_conn()` 换成 pymysql 连接、占位符 `?` 换成 `%s` 即可（文件内已标注）。

In [ ]:
# 生成交互菜单部分（核心函数与本 notebook 上面一致，部署时合并为一个 app.py）
menu_code = """
def main():
    actions = {
        "1": lambda: add_student(input("姓名："), input("班级："), input("性别："), int(input("年龄："))),
        "2": show_all,
        "3": lambda: search_by_name(input("姓名关键字：")),
        "4": lambda: search_by_cls(input("班级：")),
        "5": lambda: update_score(int(input("学号：")), input("科目："), float(input("新成绩：")),
                                  confirm=input("确认修改？(yes/n) ").strip()=="yes"),
        "6": lambda: delete_student(int(input("学号：")),
                                    confirm=input("确认删除？(yes/n) ").strip()=="yes"),
        "7": report_subject_avg,
        "8": lambda: report_top(int(input("显示前几名："))),
        "9": report_cls_compare,
        "10": lambda: print(student_report(int(input("学号：")))),
        "11": lambda: import_csv(input("CSV 路径：")),
    }
    while True:
        print("\\n===== 学生管理系统 =====")
        print("1.添加 2.全部 3.查姓名 4.查班级 5.改成绩 6.删除")
        print("7.科目统计 8.排名 9.班级对比 10.个人报告 11.导入CSV 0.退出")
        choice = input("请选择：").strip()
        if choice == "0":
            print("再见！"); break
        try:
            actions.get(choice, lambda: print("⚠️ 无效选项"))()
        except Exception as e:
            print(f"⚠️ 操作失败：{e}")

if __name__ == "__main__":
    init_db()   # 首次运行建表；已存在则跳过（生产环境用 CREATE TABLE IF NOT EXISTS）
    main()
"""
print(menu_code)
print("部署步骤：")
print("1. 把本 notebook 各函数 + 上面的 main 合并保存为 app.py")
print("2. 修改 get_conn() 为 pymysql 连接，占位符 ? 改 %s")
print("3. scp app.py ubuntu@服务器IP:~/student-mgmt/")
print("4. 服务器上 nohup python3 app.py > app.log 2>&1 &")

---
➡️ **项目完成后**：Day 16–20 进阶内容 —— [https://colab.research.google.com/github/wuqichuang/learning-python/blob/main/notebooks/07_day16-20_advanced.ipynb](https://colab.research.google.com/github/wuqichuang/learning-python/blob/main/notebooks/07_day16-20_advanced.ipynb)